# ETF volatility baselines — TF OLS version

Rebuilds the same 4 baseline columns from the earlier v2 notebook that used non-TensorFlow code:
- baseline_avg_5 and baseline_avg_20: rolling mean, no change- just arithmetic
- baseline_ols_5 and baseline_ols_20: (recoded using TF, tf.linalg.lstsq

Leakage rules are identical to v2 — everything is built from `y_known_at_t`,
which is the target shifted by HORIZON rows within each ticker.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf

# VB 041626 - same paths as v2, updated output filename so I don't overwrite the original
BASE_PATH  = '/content/drive/MyDrive/Colab Notebooks/'
DATA_PATH  = BASE_PATH + 'vol_dataset_post_wrangle_021026.csv'
OUT_PATH   = BASE_PATH + 'vol_dataset_with_4_tf_baselines_041626.csv'

DATE_COL   = 'date'
TICKER_COL = 'ticker'
Y_COL      = 'forward_vol_5d_annual_decimel_calculated'
HORIZON    = 5  # forward window in trading days

print('TF version  :', tf.__version__)
print('pandas version:', pd.__version__)

TF version  : 2.19.0
pandas version: 2.2.2


In [4]:
df0 = pd.read_csv(DATA_PATH, low_memory=False)

# crash early if required columns are missing
missing = {DATE_COL, TICKER_COL, Y_COL} - set(df0.columns)
assert not missing, f'missing cols: {missing}'

df = df0.copy()

# format='mixed' is more forgiving if date strings are not perfectly uniform
df[DATE_COL] = pd.to_datetime(df[DATE_COL], format='mixed', errors='coerce')

# IMPORTANT: sort before shifting — shifting on an unsorted df bleeds values across tickers
df = df.sort_values([TICKER_COL, DATE_COL]).reset_index(drop=True)

print(df.shape, '|', df[TICKER_COL].nunique(), 'tickers')
print(df[DATE_COL].min().date(), '->', df[DATE_COL].max().date())
print('date null count after parse:', df[DATE_COL].isna().sum())

(96525, 23) | 36 tickers
2014-12-31 -> 2025-12-31
date null count after parse: 0


## Known target at time t

`forward_vol_5d_annual_decimel_calculated` is a forward-looking column — it uses
returns from the *next* 5 days. So I shift it by HORIZON rows within each ticker
to get the version that would have been known *before* the forecast week starts.

This is the same approach as v2. Also storing it as `baseline_y_known_at_t`
so it shows up in the downstream feature set as a named column.

In [5]:
# group by ticker first — shift globally would mix rows from different tickers
df['y_known_at_t'] = df.groupby(TICKER_COL, sort=False)[Y_COL].shift(HORIZON)

# also save as a named baseline column so it's explicit in the output file
df['baseline_y_known_at_t'] = df['y_known_at_t']

# spot check — first 12 rows of first ticker
# y_known_at_t should be NaN for the first HORIZON rows, then fill in
tkr = df[TICKER_COL].dropna().iloc[0]
df[df[TICKER_COL] == tkr][[DATE_COL, Y_COL, 'y_known_at_t']].head(12)

,date,forward_vol_5d_annual_decimel_calculated,y_known_at_t
0,2021-04-21,0.0447,NaN
1,2021-04-22,0.0628,NaN
2,2021-04-23,0.1131,NaN
3,2021-04-26,0.1119,NaN
4,2021-04-27,0.1400,NaN
5,2021-04-28,0.1068,0.0447
6,2021-04-29,0.1227,0.0628
7,2021-04-30,0.2845,0.1131
8,2021-05-03,0.3002,0.1119
9,2021-05-04,0.2937,0.1400


## Rolling average baselines (avg-5, avg-20)

No change from v2 — just rolling means on `y_known_at_t` within each ticker.

In [6]:
# group_keys=False avoids a pandas FutureWarning
g = df.groupby(TICKER_COL, sort=False, group_keys=False)['y_known_at_t']

df['baseline_avg_5']  = g.rolling(5,  min_periods=5 ).mean().reset_index(level=0, drop=True)
df['baseline_avg_20'] = g.rolling(20, min_periods=20).mean().reset_index(level=0, drop=True)

df[[DATE_COL, TICKER_COL, 'baseline_avg_5', 'baseline_avg_20']].head(12)

,date,ticker,baseline_avg_5,baseline_avg_20
0,2021-04-21,BEDZ,NaN,NaN
1,2021-04-22,BEDZ,NaN,NaN
2,2021-04-23,BEDZ,NaN,NaN
3,2021-04-26,BEDZ,NaN,NaN
4,2021-04-27,BEDZ,NaN,NaN
5,2021-04-28,BEDZ,NaN,NaN
6,2021-04-29,BEDZ,NaN,NaN
7,2021-04-30,BEDZ,NaN,NaN
8,2021-05-03,BEDZ,NaN,NaN
9,2021-05-04,BEDZ,0.09450,NaN


## OLS baselines using TensorFlow (ols-5, ols-20)

This is the main change from v2. Instead of computing slope manually with NumPy,
I'm using tf.linalg.lstsq .


In [7]:
def rolling_ols_tf(series, window, horizon):
    '''
    Rolling OLS forecast using TensorFlow lstsq.

    For each rolling window of past y_known_at_t values,
    fit a straight line and extrapolate horizon
    steps ahead of the last observed point.

    series: one ticker's y_known_at_t values
    window: lookback window size (5 or 20)
    horizon: how many steps ahead to forecast
    '''
    # x-indices for the window: 0, 1, 2, ..., window-1
    x_vals = np.arange(window, dtype=np.float32)

    # forecast point is horizon steps past the last index in the window
    # e.g. window=5, horizon=5 -> x_pred = 4 + 5 = 9
    x_pred = float((window - 1) + horizon)

    def predict_one_window(y_raw):
        # pandas .apply(raw=True) passes a NumPy array
        y = y_raw.astype(np.float32).reshape(-1, 1)  # shape: (window, 1)

        # design matrix: [x_index, intercept_term] — shape (window, 2)
        X = np.column_stack([x_vals, np.ones(window, dtype=np.float32)])

        # wrap in TF constants and solve
        X_tf = tf.constant(X)    # (window, 2)
        y_tf = tf.constant(y)    # (window, 1)

        # lstsq returns coeffs shape (2, 1): index 0 = slope, index 1 = intercept
        # fast=False -> QR path, more stable than Cholesky for small windows
        coeffs = tf.linalg.lstsq(X_tf, y_tf, fast=False)

        slope     = float(coeffs[0, 0].numpy())
        intercept = float(coeffs[1, 0].numpy())

        return slope * x_pred + intercept

    return series.rolling(window, min_periods=window).apply(predict_one_window, raw=True)


print('rolling_ols_tf defined')

rolling_ols_tf defined


In [8]:
# apply per ticker — same groupby pattern as v2
g = df.groupby(TICKER_COL, sort=False, group_keys=False)['y_known_at_t']

print('computing baseline_ols_5  ...')
df['baseline_ols_5'] = (
    g.apply(lambda s: rolling_ols_tf(s, 5, HORIZON))
     .reset_index(level=0, drop=True)
)

print('computing baseline_ols_20 ...')
df['baseline_ols_20'] = (
    g.apply(lambda s: rolling_ols_tf(s, 20, HORIZON))
     .reset_index(level=0, drop=True)
)

df[[DATE_COL, TICKER_COL, 'baseline_ols_5', 'baseline_ols_20']].head(12)

computing baseline_ols_5  ...
computing baseline_ols_20 ...


,date,ticker,baseline_ols_5,baseline_ols_20
0,2021-04-21,BEDZ,NaN,NaN
1,2021-04-22,BEDZ,NaN,NaN
2,2021-04-23,BEDZ,NaN,NaN
3,2021-04-26,BEDZ,NaN,NaN
4,2021-04-27,BEDZ,NaN,NaN
5,2021-04-28,BEDZ,NaN,NaN
6,2021-04-29,BEDZ,NaN,NaN
7,2021-04-30,BEDZ,NaN,NaN
8,2021-05-03,BEDZ,NaN,NaN
9,2021-05-04,BEDZ,0.26229,NaN


## Checks

Catch anything broken before writing the output file.

In [9]:
bcols = [
    'baseline_y_known_at_t',
    'baseline_avg_5',
    'baseline_avg_20',
    'baseline_ols_5',
    'baseline_ols_20'
]

print('nan rates:')
print(df[bcols].isna().mean().round(3))

# OLS can predict negative vol
print('\nnegative OLS values (ols_5, ols_20):',
      int((df['baseline_ols_5']  < 0).sum(skipna=True)),
      int((df['baseline_ols_20'] < 0).sum(skipna=True)))

# OLS range should be in the same ballpark as avg range
print('\nrange check:')
for col in bcols[1:]:   # skip y_known_at_t, it's just the shifted target
    lo = df[col].min()
    hi = df[col].max()
    print(f'  {col}: [{lo:.4f}, {hi:.4f}]')

# deeper spot check — full column view for first ticker
check_cols = [DATE_COL, Y_COL, 'y_known_at_t',
              'baseline_avg_5', 'baseline_avg_20',
              'baseline_ols_5', 'baseline_ols_20']

print(f'\nspot check — ticker: {tkr}')
display(df[df[TICKER_COL] == tkr][check_cols].head(30))

nan rates:
baseline_y_known_at_t    0.002
baseline_avg_5           0.003
baseline_avg_20          0.009
baseline_ols_5           0.003
baseline_ols_20          0.009
dtype: float64

negative OLS values (ols_5, ols_20): 13386 1755

range check:
  baseline_avg_5: [0.0132, 3.6811]
  baseline_avg_20: [0.0305, 1.8507]
  baseline_ols_5: [-2.0199, 9.0870]
  baseline_ols_20: [-0.6477, 4.2348]

spot check — ticker: BEDZ


,date,forward_vol_5d_annual_decimel_calculated,y_known_at_t,baseline_avg_5,baseline_avg_20,baseline_ols_5,baseline_ols_20
0,2021-04-21,0.0447,NaN,NaN,NaN,NaN,NaN
1,2021-04-22,0.0628,NaN,NaN,NaN,NaN,NaN
2,2021-04-23,0.1131,NaN,NaN,NaN,NaN,NaN
3,2021-04-26,0.1119,NaN,NaN,NaN,NaN,NaN
4,2021-04-27,0.1400,NaN,NaN,NaN,NaN,NaN
5,2021-04-28,0.1068,0.0447,NaN,NaN,NaN,NaN
6,2021-04-29,0.1227,0.0628,NaN,NaN,NaN,NaN
7,2021-04-30,0.2845,0.1131,NaN,NaN,NaN,NaN
8,2021-05-03,0.3002,0.1119,NaN,NaN,NaN,NaN
9,2021-05-04,0.2937,0.1400,0.09450,NaN,0.26229,NaN


## Save output

Save to Drive.

In [10]:
df.to_csv(OUT_PATH, index=False)
print('saved:', OUT_PATH)
print('final shape:', df.shape)

saved: /content/drive/MyDrive/Colab Notebooks/vol_dataset_with_4_tf_baselines_041626.csv
final shape: (96525, 29)
